<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW5b_intercept.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW5b_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW5b: Refractor Depth from the Intercept Time

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Due:** Sunday, December 13, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) to the Brightspace HW5b dropbox

**Extra credit:** This notebook is optional. It is graded out of 100 and counts as a bonus of up to 50 points in Problem Sets & Labs.

This notebook is new, and you are the first class to work it.

The mechanics of Colab (saving, running cells, errors, uploading) are on the **Working in Colab** page in the Resources module on Brightspace.

This assignment follows the Wednesday, November 18 class (Burger sections 3.2 to 3.4). It uses a real survey over two layers and a second survey over three layers. The **refractor** is the layer boundary the head wave travels along: the top of a faster layer.

## What you will do

1. Run the HW5a load and fit on a new survey, as given code.
2. Find the thickness of the upper layer from the intercept time and from the crossover distance.
3. Find the critical distance.
4. Fit a second survey over three layers and find two thicknesses.

## What you will hand in

This same notebook with every cell run and the answer cells filled in. Each question has one answer cell that opens with its label. Double-click the cell and type under the label.

## Feedback

After each section there is a feedback cell, and one more at the end. Tell me what worked, what did not, and where you got stuck. Write it as you go, while it is fresh. The feedback cells carry no points.

## Learning objectives

This notebook builds these course objectives:

6. Compute the depth to a flat refractor from the intercept time. Compute it again from the crossover distance. Explain why the intercept time usually gives the better value.
7. Compute the critical distance from the depth and the two velocities. Explain why no head wave arrives closer to the source than the critical distance. Tell the critical distance apart from the crossover distance.
8. Find the three velocities from the three lines of a three-layer plot. Compute the thickness of the first layer. Then use it with the second intercept time to compute the thickness of the second layer.

## Data source and citation

The first survey is a real field record. The second is described in the three-layer section below.

> Burton, B.L., Hunter, L.E., Cox, E.M., Lindsay, K.M., Galloway, R.G., and Nguyen, H., 2020, Success Dam Spillway seismic refraction survey, Porterville, California, December 2018 and July 2019: U.S. Geological Survey data release, https://doi.org/10.5066/P96U1EYD. Public domain (U.S. Government work).

The U.S. Geological Survey shot this line in the Success Dam spillway near Porterville, California. The file holds one shot: 33 geophones, 1.83 m (6 ft) apart, from 0.91 m to 59.44 m. The source is a weight driven into the ground from a truck (an accelerated weight drop) at 0 m. The USGS picked the time of the first ground motion at each geophone. That time is the [**first break**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#first-break). The course took this one shot from the published picks and converted the distances from feet to meters.

| Column | Meaning |
|---|---|
| `offset_m` | Distance from the source to the geophone (m) |
| `first_break_ms` | First-break time (ms) |
| `travel_time_s` | The same time in seconds |

## Loading the data

The code below loads `refraction_success_dam_l4_s4041.csv` and `refraction_three_layer.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download both files from the Brightspace HW5b page.
2. In Google Colab, click the **Files** icon in the left sidebar.
3. Drag both files into the file panel.
4. In each loading cell below, comment out the `pd.read_csv` line that reads a web address and use the commented line that reads the file by name.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports load NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

The cell also defines `run_check`. A run-check cell prints one line that reads PASS or FAIL. A run check carries no question and no points. It confirms that the cells above ran as written. If one reads FAIL, the problem is upstream of it.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe
COLORS = px.colors.qualitative.Safe


def run_check(name, passed, detail):
    """Print one run-check line: PASS, or FAIL with the value that missed."""
    print(f'{name} {"PASS" if passed else "FAIL"}: {detail}')


## The HW5a load and fit, on a new survey

HW5a fitted one straight line to the direct-wave first breaks and a second to the head-wave first breaks. The cells below run the same fit on the real survey, with the split distance `LAST_DIRECT_M` given. They are given code with no questions attached. Run them in order.

**Load.** Reads `refraction_success_dam_l4_s4041.csv` into `df` and defines `x`, the distance from the source, and `t`, the first-break time in seconds.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/refraction_success_dam_l4_s4041.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download
# refraction_success_dam_l4_s4041.csv from the Brightspace HW5b page, drag it into
# the Colab file panel, then comment out the df = pd.read_csv(DATA_URL) line above
# and use:
# df = pd.read_csv("refraction_success_dam_l4_s4041.csv")

x = df['offset_m'].to_numpy()          # distance from the source (m)
t = df['travel_time_s'].to_numpy()     # first-break time (s)
print(f'{len(df)} geophones, {x.min():.2f} m to {x.max():.2f} m, '
      f'{np.diff(x).min():.2f} m apart')
df.head(10)


**Fit the two lines.** The geophones up to `LAST_DIRECT_M` (10.1 m) are on the direct-wave line. The rest are on the head-wave line. Each velocity is one over its slope. The cell prints both velocities, the critical angle and the distance where the two fitted lines cross.

In [ ]:
LAST_DIRECT_M = 10.1   # distance of the last geophone on the steep (direct-wave) line, in meters

direct = x <= LAST_DIRECT_M
head = x > LAST_DIRECT_M

# One straight line through each group.
slope_d, intercept_d = np.polyfit(x[direct], t[direct], 1)
slope_h, intercept_h = np.polyfit(x[head], t[head], 1)

# Velocity is one over the slope.
v1 = 1.0 / slope_d
v2 = 1.0 / slope_h

theta_ic = np.degrees(np.arcsin(v1 / v2))                  # critical angle, Burger eq 2.27
x_co = (intercept_h - intercept_d) / (slope_d - slope_h)   # where the two lines give the same time

print(f'geophones on the direct line: {direct.sum()}; on the head-wave line: {head.sum()}')
print(f'direct-wave slope: {slope_d * 1000:.3f} ms per m   V1 = {v1:.0f} m/s')
print(f'head-wave slope:   {slope_h * 1000:.3f} ms per m   V2 = {v2:.0f} m/s')
print(f'critical angle:    {theta_ic:.1f} degrees')
print(f'the lines cross at {x_co:.1f} m')


This cell plots the first breaks with the two fitted lines.

In [ ]:
x_line = np.linspace(0.0, x.max(), 200)
fig = go.Figure()
fig.add_scatter(x=x, y=t * 1000, mode='markers', name='first breaks',
                marker=dict(size=9, symbol='circle', color=COLORS[0]))
fig.add_scatter(x=x_line, y=(slope_d * x_line + intercept_d) * 1000, mode='lines',
                name=f'direct-wave line, V1 = {v1:.0f} m/s',
                line=dict(color=COLORS[3], width=2, dash='solid'))
fig.add_scatter(x=x_line, y=(slope_h * x_line + intercept_h) * 1000, mode='lines',
                name=f'head-wave line, V2 = {v2:.0f} m/s',
                line=dict(color=COLORS[4], width=2, dash='dash'))
fig.add_vline(x=x_co, line_dash='dot', annotation_text=f'lines cross at {x_co:.1f} m')
fig.update_xaxes(title_text='Distance from the source (m)')
fig.update_yaxes(title_text='First-break time (ms)', rangemode='tozero')
fig.update_layout(title='First breaks with the two fitted lines')
fig.show()


**Figure description:** The first breaks as circles, first-break time (ms, y-axis) against distance from the source (m, x-axis), with two fitted lines drawn from 0 m to 59 m. A solid line runs through the near points and meets the time axis at about 1.5 ms. A dashed line runs through the far points and meets the time axis at about 16 ms. The solid line is the steeper of the two. A dotted vertical line marks where the two lines cross, near 10 m. Non-visual path: the previous cell prints both slopes, both velocities and the crossing distance.

The direct-wave line misses the origin by about 1.5 ms.

**Run check 1.** $V_1$ should fall between 470 and 570 m/s and $V_2$ between 1980 and 2130 m/s. The lines should cross between 9 m and 11 m. If this reads FAIL, the cells above did not run as written: choose Runtime > Restart and run all.

In [ ]:
run_check('Run check 1',
          470 <= v1 <= 570 and 1980 <= v2 <= 2130 and 9 <= x_co <= 11,
          f'V1 {v1:.0f} m/s (470 to 570); V2 {v2:.0f} m/s (1980 to 2130); '
          f'lines cross at {x_co:.1f} m (9 to 11)')


**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 1:**



## Layer thickness from intercept times and the crossover distance

This section starts from the fit above.

### Two layers

Extend the head-wave line back to $x = 0$. The time where it meets the time axis is the **intercept time**, $t_i$. No head wave arrives at the source, so $t_i$ is not a measured arrival. It is a property of the line. Burger solves it for the thickness of the upper layer (eq 3.17):

$$h_1 = \frac{t_i}{2} \frac{V_2 V_1}{\sqrt{V_2^2 - V_1^2}}$$

At the crossover distance $x_{co}$ the two waves arrive at the same time. Setting the two travel times equal and solving for $h_1$ gives a second route to the thickness (eq 3.21):

$$h_1 = \frac{x_{co}}{2} \sqrt{\frac{V_2 - V_1}{V_2 + V_1}}$$

The head wave first reaches the surface at the [**critical distance**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#critical-distance), $x_{crit}$, where the ray at the critical angle comes back up (Fig 3.5, eq 3.25):

$$x_{crit} = \frac{2 h_1}{\sqrt{(V_2 / V_1)^2 - 1}}$$

**Thickness two ways.** This cell takes $t_i$ from the head-wave fit above. It computes $h_1$ from eq 3.17 and from eq 3.21, then the critical distance from eq 3.25.

In [ ]:
# The intercept time is the head-wave line at x = 0, from the fit above.
t_i = intercept_h

h1_from_ti = (t_i / 2) * v2 * v1 / np.sqrt(v2**2 - v1**2)     # Burger eq 3.17
h1_from_xco = (x_co / 2) * np.sqrt((v2 - v1) / (v2 + v1))     # Burger eq 3.21
x_crit = 2 * h1_from_ti / np.sqrt((v2 / v1)**2 - 1)           # Burger eq 3.25

print(f'intercept time t_i = {t_i * 1000:.1f} ms')
print(f'thickness of the upper layer from t_i (eq 3.17):  {h1_from_ti:.1f} m')
print(f'thickness of the upper layer from x_co (eq 3.21): {h1_from_xco:.1f} m')
print(f'critical distance (eq 3.25): {x_crit:.1f} m')
print(f'crossover distance (fit above): {x_co:.1f} m')


**Run check 2.** Both thicknesses should fall between 3.5 and 4.5 m, and the critical distance between 1.8 and 2.6 m. If this reads FAIL, choose Runtime > Restart and run all.

In [ ]:
run_check('Run check 2',
          3.5 <= h1_from_ti <= 4.5 and 3.5 <= h1_from_xco <= 4.5 and 1.8 <= x_crit <= 2.6,
          f'thickness {h1_from_ti:.1f} m and {h1_from_xco:.1f} m (3.5 to 4.5); '
          f'critical distance {x_crit:.1f} m (1.8 to 2.6)')


**Question 1.** Report $t_i$. Find the thickness of the upper layer two ways: from the intercept time, and from the crossover distance. Burger prefers one of the two (section 3.2.4). Which one, and why? Hint: you read $t_i$ where one line meets the time axis. You read the crossover where two lines meet each other.

**Q1 answer:**



**Question 2.** Report the critical distance. Head waves reach every geophone beyond it. Between the critical distance and the crossover near 10 m, which wave gives the first break, and why? (Burger section 3.2.5, Fig 3.5 and Table 3.1.)

**Q2 answer:**



**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 2:**



### Three layers

A second survey was shot over three flat layers. Each layer is faster than the one above it. A third arrival now competes for the first break: the head wave along the top of layer 3. Its line has slope $1/V_3$ (eq 3.34).

The intercept time of the third line, $t_{i2}$, holds the time the wave spends crossing layer 1 and crossing layer 2, on the way down and back up. Subtract the layer 1 part, and the rest gives the thickness of layer 2 (eq 3.35):

$$h_2 = \left(t_{i2} - \frac{2 h_1 \sqrt{V_3^2 - V_1^2}}{V_3 V_1}\right) \frac{V_3 V_2}{2 \sqrt{V_3^2 - V_2^2}}$$

The first intercept time, $t_{i1}$, gives $h_1$ by eq 3.17, the same as in the two-layer case. Thicknesses come out from the top down (section 3.4).

> Second dataset (synthetic): generated by a course script (GEOL 333/714 course materials, seed 333). Three layers over flat interfaces, with a small random error added to each time. No external license.

The survey is one line of 48 geophones, 2 m apart, from 2 m to 96 m, with the source at 0 m. The columns are the same as in HW5a.

**Load.** Reads `refraction_three_layer.csv` into `df3` and prints every geophone as a table. The names `x_b` and `t_b` keep this survey apart from the first survey.

In [ ]:
DATA_URL_3 = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/refraction_three_layer.csv"
df3 = pd.read_csv(DATA_URL_3)

# Fallback (no internet, or URL not live yet): download refraction_three_layer.csv
# from the Brightspace HW5b page, drag it into the Colab file panel, then comment
# out the df3 = pd.read_csv(DATA_URL_3) line above and use:
# df3 = pd.read_csv("refraction_three_layer.csv")

# The _b names keep this survey apart from the first survey.
x_b = df3['offset_m'].to_numpy()          # distance from the source (m)
t_b = df3['travel_time_s'].to_numpy()     # first-break time (s)
print(f'{len(df3)} geophones, {x_b.min():.0f} m to {x_b.max():.0f} m, '
      f'{np.diff(x_b).min():.0f} m apart')

# Every geophone as a table.
print(df3[['offset_m', 'first_break_ms']].to_string(index=False))


**Fit the three lines.** `LAST_DIRECT_M_B` and `LAST_MIDDLE_M_B` mark the last geophone on the first and second lines. The cell fits a line to each group and gives each velocity as one over its slope. The intercepts of the second and third lines are $t_{i1}$ and $t_{i2}$. The cell then computes $h_1$ from eq 3.17 and $h_2$ from eq 3.35.

In [ ]:
LAST_DIRECT_M_B = 16.0   # last geophone on the steepest line (m)
LAST_MIDDLE_M_B = 46.0   # last geophone on the middle line (m)

seg1 = x_b <= LAST_DIRECT_M_B
seg2 = (x_b > LAST_DIRECT_M_B) & (x_b <= LAST_MIDDLE_M_B)
seg3 = x_b > LAST_MIDDLE_M_B

s1, b1 = np.polyfit(x_b[seg1], t_b[seg1], 1)
s2, t_i1 = np.polyfit(x_b[seg2], t_b[seg2], 1)
s3, t_i2 = np.polyfit(x_b[seg3], t_b[seg3], 1)

v1_b, v2_b, v3_b = 1 / s1, 1 / s2, 1 / s3

# Thickness of layer 1 from the first intercept time (Burger eq 3.17).
h1_b = (t_i1 / 2) * v2_b * v1_b / np.sqrt(v2_b**2 - v1_b**2)

# Thickness of layer 2 from the second intercept time (Burger eq 3.35).
# The first term inside the parentheses is the part of t_i2 spent in layer 1.
layer1_part = 2 * h1_b * np.sqrt(v3_b**2 - v1_b**2) / (v3_b * v1_b)
h2_b = (t_i2 - layer1_part) * v3_b * v2_b / (2 * np.sqrt(v3_b**2 - v2_b**2))

print(f'geophones per line: {seg1.sum()}, {seg2.sum()}, {seg3.sum()}')
print(f'V1 = {v1_b:.0f} m/s   V2 = {v2_b:.0f} m/s   V3 = {v3_b:.0f} m/s')
print(f'intercept times: t_i1 = {t_i1 * 1000:.1f} ms, t_i2 = {t_i2 * 1000:.1f} ms')
print(f'part of t_i2 spent in layer 1: {layer1_part * 1000:.1f} ms')
print(f'h1 = {h1_b:.1f} m   h2 = {h2_b:.1f} m   depth to the third layer = {h1_b + h2_b:.1f} m')


In [ ]:
x_line = np.linspace(0.0, x_b.max(), 200)
fig = go.Figure()
fig.add_scatter(x=x_b, y=t_b * 1000, mode='markers', name='first breaks',
                marker=dict(size=8, symbol='circle', color=COLORS[0]))
fig.add_scatter(x=x_line, y=(s1 * x_line + b1) * 1000, mode='lines',
                name=f'{v1_b:.0f} m/s', line=dict(color=COLORS[3], width=2, dash='solid'))
fig.add_scatter(x=x_line, y=(s2 * x_line + t_i1) * 1000, mode='lines',
                name=f'{v2_b:.0f} m/s', line=dict(color=COLORS[4], width=2, dash='dash'))
fig.add_scatter(x=x_line, y=(s3 * x_line + t_i2) * 1000, mode='lines',
                name=f'{v3_b:.0f} m/s', line=dict(color=COLORS[1], width=2, dash='dot'))
fig.update_xaxes(title_text='Distance from the source (m)')
fig.update_yaxes(title_text='First-break time (ms)', rangemode='tozero')
fig.update_layout(title='Second survey with the three fitted lines')
fig.show()


**Figure description:** The first breaks of the second survey as circles, first-break time (ms, y-axis) against distance from the source (m, x-axis), with three fitted lines drawn from 0 m to 96 m. The times run from about 4 ms at 2 m to about 66 ms at 96 m. The points fall on three straight lines, each shallower than the one before: 2 m to 16 m, 18 m to 46 m, and 48 m to 96 m. The solid line through the near points starts at zero time. The dashed line through the middle points meets the time axis at about 21 ms. The dotted line through the far points meets it at about 43 ms and is the shallowest. Non-visual path: the load cell prints every geophone, and the fit cell prints all three velocities, both intercept times and both thicknesses.

**Run check 3.** $V_1$ should fall between 470 and 530 m/s, $V_2$ between 1300 and 1650 m/s and $V_3$ between 3500 and 4600 m/s. The thickness $h_1$ should fall between 5 and 7 m and $h_2$ between 13.5 and 16.5 m. If this reads FAIL, choose Runtime > Restart and run all.

In [ ]:
run_check('Run check 3',
          470 <= v1_b <= 530 and 1300 <= v2_b <= 1650 and 3500 <= v3_b <= 4600
          and 5 <= h1_b <= 7 and 13.5 <= h2_b <= 16.5,
          f'V1 {v1_b:.0f}, V2 {v2_b:.0f}, V3 {v3_b:.0f} m/s; h1 {h1_b:.1f} m (5 to 7); '
          f'h2 {h2_b:.1f} m (13.5 to 16.5)')


**Question 3.** Report the three velocities and the two thicknesses. How deep is the top of layer 3 below the source? Eq 3.35 needs $h_1$ before it can give $h_2$. Explain why from the ray path of the wave that travels along the top of layer 3. (Burger sections 3.3 and 3.4.)

**Q3 answer:**



**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 3:**



**Feedback on the whole notebook.** What would you change before another class works this notebook?

**Feedback 4:**



## How to submit

1. Run all cells from the top. (`Runtime > Run all` in Colab.) Every run-check line reads PASS.
2. Make sure every answer cell is filled in.
3. `File > Download > Download .ipynb`.
4. Rename the file to `HW5b_LASTNAME.ipynb` (for example `HW5b_Smith.ipynb`).
5. Upload it to the Brightspace HW5b dropbox by **Sunday, December 13, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.